# W2 Practical (Thursday): From diagram to model with SIP

**Week 2 · Complex Systems (MSc CSP) · Thursday session — *run the production pipeline***

On **Tuesday** you built each model by hand, writing every rate function yourself. In real projects we
rarely know the numbers up front — we know the **structure**: which things push which, and in which
direction. **SIP** (the *Systems Insight Pipeline*, the **Diagrams-to-Dynamics / D2D** method) takes
exactly that — a **signed causal-loop diagram** drawn in **Kumu** and exported to **Excel** — samples
the unknown link strengths, simulates the model hundreds of times, and tells you **which intervention
moves your outcome most**.

> **The workflow:** *structure in **Kumu** → numbers & equations in **Excel** → **SIP** builds and
> runs the model.* (Kumu has no equation field; SIP reads the Excel export — see example excel.)

**Today's research questions.** *Given only the signs of the links, which policy lever raises
adoption most — and how sure can we be?*

*By the end you can:* load a CLD into SIP · run it under parameter uncertainty (D2D) · read an
**intervention ranking** · and judge how **robust** that ranking is. Each section has an **Idea** and a
**Look for**; numbered **TODOs** have hints and expected results.

### Set up SIP
[SIP](https://github.com/vvvasconcelos/SystemsInsightPipeline) — the *Systems Insight Pipeline* — turns a Kumu causal-loop diagram (exported to **Excel**) into a runnable system-dynamics model. The cell below installs it **only if it isn't already available**, so it works the same on Google Colab and on a laptop. Run it once and wait for the `SIP ... ready` message.

In [ ]:
# Idempotent, Colab-safe: import SIP, installing the course-pinned version if missing.
SIP_PIN = "v0.4.0"  # course-pinned SIP version
SIP_GIT = "git+https://github.com/vvvasconcelos/SystemsInsightPipeline.git"
try:
    import sip_systemsinsightpipeline as sip
except ImportError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"{SIP_GIT}@{SIP_PIN}"],
        check=True,
    )
    import sip_systemsinsightpipeline as sip
print("SIP", sip.__version__, "ready")

# If SIP was ALREADY importable, the install above never ran - so check the version
# we actually got. A stale SIP silently produces different numbers from the ones
# this notebook expects.
if sip.__version__ != SIP_PIN.lstrip("v"):
    print("\n!! WARNING: this notebook expects SIP", SIP_PIN,
          "but version", sip.__version__, "is installed.")
    print("   The expected values printed in this notebook may not reproduce.")
    print("   Fix:  %pip install --force-reinstall --no-deps " + SIP_GIT + "@" + SIP_PIN)
    print("   ...then restart the kernel and re-run.")

---
## 0. Setup
Run this after the SIP cell above.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

for _b in [Path.cwd(), *Path.cwd().parents]:
    _c = _b / "templates" / "polder.mplstyle"
    if _c.exists():
        plt.style.use(str(_c)); break

TEAL, DARK_TEAL, CORAL, SLATE = "#00A39B", "#006661", "#E8633A", "#5F6B7A"
print("Setup OK.")

---
## 1. The model: a sustainable-heating adoption CLD

**Idea.** This is the same *limits-to-growth* structure you simulated on Tuesday, drawn as a CLD:
a **reinforcing** loop (a larger *installed base* raises **social proof**, which raises adoption) and a
**balancing** loop (a larger *installed base* shrinks the **remaining pool of potential adopters**, which
slows adoption). Three policy **levers** enter by
different paths — a **Subsidy** (via perceived **cost**), an **Information campaign** (via social
proof), and **Installer capacity** (directly). The outcome (VOI) is *Households on the standard*.

The diagram lives in `adoption_cld.xlsx` (the Kumu→Excel export): an **Elements** sheet (each variable
+ its type: *Stock / Auxiliary / Constant*; `Tags≠0` marks a lever) and a **Connections** sheet
(signed links). **Look for:** a teal `+` and a coral `−` link, and the two feedback loops.

In [ ]:
# Draw the CLD straight from the Excel, so you can read the structure SIP will use.
import networkx as nx
elem = pd.read_excel("adoption_cld.xlsx", sheet_name="Elements")
conn = pd.read_excel("adoption_cld.xlsx", sheet_name="Connections")
print("ELEMENTS\n", elem.to_string(index=False))
print("\nCONNECTIONS\n", conn[["From", "To", "Type"]].to_string(index=False))

short = {"Households on the standard": "Adopters", "Social proof": "Social\nproof",
         "Remaining potential adopters": "Pool", "Perceived cost": "Cost",
         "Subsidy": "Subsidy", "Information campaign": "Campaign", "Installer capacity": "Installers"}
pos = {"Adopters": (0, 0), "Social\nproof": (-1.4, 1.0), "Pool": (1.4, 1.0),
       "Cost": (0, -1.5), "Subsidy": (-1.7, -1.5), "Campaign": (-2.4, 1.0), "Installers": (2.4, 0)}
G = nx.DiGraph()
for _, row in conn.iterrows():
    G.add_edge(short[row["From"]], short[row["To"]], sign=row["Type"])
fig, ax = plt.subplots(figsize=(8.5, 6))
kinds = {short[r.Label]: r.Type for r in elem.itertuples(index=False)}
ncol = {"Stock": TEAL, "Auxiliary": "#9FE1CB", "Constant": "#E6A700"}
nx.draw_networkx_nodes(G, pos, node_size=2600,
                       node_color=[ncol[kinds[n]] for n in G.nodes], edgecolors=DARK_TEAL, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
for u, v, d in G.edges(data=True):
    col = TEAL if d["sign"] == "+" else CORAL
    ax.annotate("", xy=pos[v], xytext=pos[u],
                arrowprops=dict(arrowstyle="-|>", color=col, lw=2,
                                connectionstyle="arc3,rad=0.16", shrinkA=26, shrinkB=26))
ax.set_title("Adoption CLD: teal=+ link, coral=- link  (Stock=teal, Auxiliary=mint, Constant=amber)",
             color=DARK_TEAL, fontsize=10)
ax.axis("off"); plt.tight_layout(); plt.show()

**SIP reads the same file.** `Extract(...).extract_settings()` classifies the
variables, finds the feedback loops, and confirms each loop passes through a **stock** (the structural
reason a qualitative loop can be simulated).

In [ ]:
from sip_systemsinsightpipeline import Extract, SDM

s = Extract("adoption_cld.xlsx").extract_settings()
print("Variable of interest :", s.variable_of_interest)
print("Interventions (levers):", s.intervention_variables)
print("Stocks    :", s.stocks)
print("Auxiliaries:", s.auxiliaries)
print("Constants :", s.constants)
# SIP reports the feedback loops it found and checks every loop runs through a stock.

---
## 2. Run it under uncertainty (the D2D method)

**Idea.** The diagram gives signs, not strengths. So SIP **samples** each link strength `N` times and
simulates the system for every draw — turning "we don't know the numbers" into an honest *distribution*
of outcomes. You set the horizon and how wide the sampling is.

**Look for:** for each lever, a band of simulated effects on the outcome — the spread *is* the
uncertainty from not knowing the link strengths.

In [ ]:
# Tell SIP how to explore the unknown link strengths, then simulate.
s.seed = 7
s.N = 200
s.t_end = 25
s.time_unit = 'years'
s.parameter_value_aux = ???    # TODO: sampling bound for auxiliary links, e.g. 0.1
s.parameter_value_stocks = ??? # TODO: sampling bound for stock links, e.g. 0.1

sdm = SDM(s)
df_sol, params = sdm.run_simulations()
effects = sdm.get_intervention_effects()
voi = s.variable_of_interest[0]
lev = effects[voi]

# boxplot of each lever's effect distribution (provided)
names = list(lev)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
bp = ax.boxplot([lev[n] for n in names], vert=False, tick_labels=names,
                patch_artist=True, medianprops=dict(color=DARK_TEAL, lw=2))
for box in bp['boxes']: box.set(facecolor='#9FE1CB', alpha=.75)
ax.axvline(0, color=SLATE, ls=':', lw=1)
ax.set_xlabel(f"Effect on '{voi}'"); ax.set_title('Effect of each lever under uncertainty', color=DARK_TEAL)
plt.tight_layout(); plt.show()

---
## 3. Which lever matters most?

**Idea.** Collapse each lever's distribution to a ranking: the **median effect** on the outcome, with
its uncertainty. This is the D2D pay-off — *leverage from a map alone*.

**Look for:** one lever clearly ahead; note whether the gap is bigger than the spread.

In [ ]:
# Collapse each lever's distribution to a ranking: median effect + a 5-95% interval.
med = {k: float(np.median(v)) for k, v in lev.items()}
lo  = {k: float(np.percentile(v, 5))  for k, v in lev.items()}
hi  = {k: float(np.percentile(v, 95)) for k, v in lev.items()}
order = sorted(med, key=med.get)               # ascending -> strongest lever on top

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.barh(range(len(order)), [med[k] for k in order], color=TEAL, height=.62,
        xerr=[[med[k] - lo[k] for k in order], [hi[k] - med[k] for k in order]],
        error_kw=dict(ecolor=SLATE, lw=1.4, capsize=4))
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.axvline(0, color=SLATE, ls=":", lw=1)
ax.set_xlabel(f"Median effect on '{voi}'  (bar = 5-95% interval)")
ax.set_title("Which lever matters most?", color=DARK_TEAL)
plt.tight_layout(); plt.show()

print("Ranking (strongest first):")
for k in sorted(med, key=med.get, reverse=True):
    print(f"   {k:22s} median {med[k]:+.3f}   [5-95%: {lo[k]:+.3f}, {hi[k]:+.3f}]")
print("\n(SIP also ships plots.plot_simulated_intervention_ranking for a built-in version.)")

---
## 4. Is the ranking robust?

**Idea.** A ranking from one set of assumptions can mislead. Widen the uncertainty and change the seed:
a *robust* conclusion survives; a fragile one flips.

In [ ]:
# TODO 1 - How robust is the ranking?
# 1. widen uncertainty: s.parameter_value_aux = 0.3; s.parameter_value_stocks = 0.3; s.seed = 99
# 2. sdm2 = SDM(s); df2,_ = sdm2.run_simulations(); eff2 = sdm2.get_intervention_effects()
# 3. print the median effects and compare the order with section 3.
#    (expected: the strongest lever is usually stable; spreads widen, close levers can swap)
s.parameter_value_aux = ???
s.parameter_value_stocks = ???
s.seed = 99
# ... build sdm2, run, and print median effects per lever

---
## 5. The real workflow: change the diagram

**Idea.** Improving a model means changing the **structure** — adding a node and a link — then
re-running. The supported route is editing the **Excel** export (*structure in Kumu, numbers/equations
in Excel*). Here we add a **Mandate** lever programmatically so the notebook stays runnable; in your own
work you would add the row in Excel and re-load.

In [ ]:
# TODO 2 - Add a 'Mandate' lever, then re-rank.
# Edit the diagram via the Excel export (structure in Kumu, numbers in Excel).
import openpyxl
wb = openpyxl.load_workbook('adoption_cld.xlsx')
# TODO: append a Mandate row to 'Elements' (Constant, Tag 1) and a '+' link
#       Mandate -> Households on the standard to 'Connections', then save as adoption_cld_v2.xlsx
wb.save('adoption_cld_v2.xlsx')
# then: re-Extract adoption_cld_v2.xlsx, set the same settings, run, and print the new ranking

---
## Bridge & exam pointers

You took a **qualitative** CLD — only link signs — and got a **ranked set of interventions under
uncertainty**, without ever choosing the numbers. That is the Diagrams-to-Dynamics idea, and the
connective tool for the rest of the course:

- **W3** quantifies the *loopless* links from data (so the strengths stop being guesses).
- **W6** returns to this exact engine for **global sensitivity** (`run_GSA` — Sobol / delta / PAWN) and
  **scenario discovery** (`discover_scenarios` — PRIM / CART), and asks you to design the pipeline yourself.

*Examinable:* reading a CLD's loops (R/B); why every loop needs a stock; what an intervention ranking
under uncertainty does and does **not** tell you (it ranks *given the structure*; it cannot invent
strengths the data could pin down).

## What you should now be able to do
- ✓ Export a CLD as the SIP **Excel** format and load it with `Extract`
- ✓ Run a model **under parameter uncertainty** (D2D) and read the ensemble
- ✓ Produce and interpret an **intervention ranking**
- ✓ Test the **robustness** of that ranking, and change the model by editing the diagram